# Tutorial 12: Advanced Workflows

**Duration:** 30-40 minutes

This tutorial demonstrates complete end-to-end analysis workflows combining multiple spatialtissuepy modules.

## Learning Objectives

By the end of this tutorial, you will be able to:
- Design comprehensive spatial analysis pipelines
- Combine multiple analysis methods for biological insight
- Create reproducible analysis workflows
- Generate publication-ready outputs

## Prerequisites

- Tutorials 1-11 completed
- Understanding of all spatialtissuepy modules

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Core
from spatialtissuepy import SpatialTissueData
from spatialtissuepy.io import read_csv, write_json

# Analysis modules
from spatialtissuepy.spatial.neighborhood import (
    compute_neighborhoods,
    neighborhood_composition,
    neighborhood_diversity,
    interface_cells,
)
from spatialtissuepy.statistics import (
    ripleys_h,
    colocalization_matrix,
    getis_ord_gi_star,
)
from spatialtissuepy.network import CellGraph, type_assortativity, centrality_by_type
from spatialtissuepy.lda import fit_spatial_lda, topic_enrichment, dominant_topic_per_cell
from spatialtissuepy.topology import spatial_mapper, extract_mapper_features
from spatialtissuepy.summary import StatisticsPanel, MultiSampleSummary

# Visualization
from spatialtissuepy.viz import (
    set_publication_style,
    plot_spatial_scatter,
    plot_ripleys_curve,
    plot_colocalization_heatmap,
    plot_topic_spatial,
    plot_mapper_graph,
    save_figure,
)

np.random.seed(42)

## Workflow 1: Complete Tumor Microenvironment Analysis

This workflow analyzes a single tissue sample comprehensively.

In [ ]:
# Create realistic TME sample
def create_tme_sample():
    """Create a tumor microenvironment sample."""
    # Tumor core
    tumor_core = np.random.normal([500, 500], 80, (180, 2))
    
    # Tumor periphery (more spread)
    tumor_periph = np.random.normal([500, 500], 150, (60, 2))
    tumor_coords = np.vstack([tumor_core, tumor_periph])
    
    # CD8 T cells (infiltrating)
    cd8_coords = np.random.normal([500, 500], 180, (80, 2))
    
    # CD4 T cells (helper)
    cd4_coords = np.random.normal([500, 500], 200, (50, 2))
    
    # Macrophages (TAMs)
    theta = np.random.uniform(0, 2*np.pi, 60)
    r = np.random.normal(150, 30, 60)
    macro_coords = np.column_stack([500 + r*np.cos(theta), 500 + r*np.sin(theta)])
    
    # Stromal cells
    stromal_coords = np.random.uniform(0, 1000, (80, 2))
    
    # Combine
    coords = np.vstack([tumor_coords, cd8_coords, cd4_coords, macro_coords, stromal_coords])
    types = (
        ['Tumor'] * len(tumor_coords) +
        ['CD8_T_cell'] * len(cd8_coords) +
        ['CD4_T_cell'] * len(cd4_coords) +
        ['Macrophage'] * len(macro_coords) +
        ['Stromal'] * len(stromal_coords)
    )
    
    # Add markers
    n = len(coords)
    markers = pd.DataFrame({
        'PD1': np.random.lognormal(1.5, 0.8, n),
        'PDL1': np.random.lognormal(1.0, 0.5, n),
        'Ki67': np.random.uniform(0, 100, n),
        'CD68': np.random.lognormal(1.0, 0.8, n),
    })
    
    return SpatialTissueData(coordinates=coords, cell_types=np.array(types), markers=markers)

tissue = create_tme_sample()
print(tissue)

### Step 1: Initial Visualization and QC

In [ ]:
# QC summary
print("=" * 50)
print("TISSUE SAMPLE QC")
print("=" * 50)
print(f"\nTotal cells: {tissue.n_cells}")
print(f"Cell types: {len(tissue.cell_types_unique)}")
print(f"Markers: {list(tissue.markers.columns)}")
print(f"\nCell type distribution:")
for ct in tissue.cell_types_unique:
    n = np.sum(tissue.cell_types == ct)
    pct = 100 * n / tissue.n_cells
    print(f"  {ct}: {n} ({pct:.1f}%)")

In [ ]:
# Initial visualization
fig, ax = plt.subplots(figsize=(10, 10))
plot_spatial_scatter(tissue, ax=ax)
ax.set_title('Tumor Microenvironment Sample')
plt.tight_layout()
plt.show()

### Step 2: Spatial Statistics

In [ ]:
print("\n" + "=" * 50)
print("SPATIAL STATISTICS")
print("=" * 50)

# Ripley's H for each cell type
radii = np.linspace(10, 150, 15)
print("\nRipley's H at r=100μm (>0 = clustered, <0 = dispersed):")
for ct in tissue.cell_types_unique:
    mask = tissue.cell_types == ct
    coords = tissue.coordinates[mask]
    if len(coords) >= 10:
        H = ripleys_h(coords, radii=radii)
        h_100 = H[np.argmin(np.abs(radii - 100))]
        status = "clustered" if h_100 > 10 else "dispersed" if h_100 < -10 else "random"
        print(f"  {ct}: H={h_100:.1f} ({status})")

In [ ]:
# Colocalization analysis
clq = colocalization_matrix(tissue, radius=50.0)

print("\nColocalization Quotient (radius=50μm):")
print("(>1 = attract, <1 = repel)")
print(clq.round(2))

### Step 3: Neighborhood Analysis

In [ ]:
print("\n" + "=" * 50)
print("NEIGHBORHOOD ANALYSIS")
print("=" * 50)

# Compute neighborhoods
neighborhoods = compute_neighborhoods(tissue, method='radius', radius=50.0)

# Diversity
diversity = neighborhood_diversity(tissue, neighborhoods, metric='shannon')
print(f"\nMean neighborhood diversity (Shannon): {diversity.mean():.3f}")
print("By cell type:")
for ct in tissue.cell_types_unique:
    mask = tissue.cell_types == ct
    print(f"  {ct}: {diversity[mask].mean():.3f}")

In [ ]:
# Interface analysis
tumor_int, cd8_int = interface_cells(tissue, 'Tumor', 'CD8_T_cell', radius=50.0)

print(f"\nTumor-CD8 Interface:")
print(f"  Tumor cells at interface: {len(tumor_int)} / {np.sum(tissue.cell_types == 'Tumor')} ({100*len(tumor_int)/np.sum(tissue.cell_types == 'Tumor'):.1f}%)")
print(f"  CD8 cells at interface: {len(cd8_int)} / {np.sum(tissue.cell_types == 'CD8_T_cell')} ({100*len(cd8_int)/np.sum(tissue.cell_types == 'CD8_T_cell'):.1f}%)")

### Step 4: Network Analysis

In [ ]:
print("\n" + "=" * 50)
print("NETWORK ANALYSIS")
print("=" * 50)

# Build graph
graph = CellGraph.from_spatial_data(tissue, method='proximity', radius=50.0)

print(f"\nGraph statistics:")
print(f"  Nodes: {graph.n_nodes}")
print(f"  Edges: {graph.n_edges}")
print(f"  Mean degree: {graph.mean_degree:.2f}")

# Assortativity
assort = type_assortativity(graph)
print(f"\nType assortativity: {assort:.3f}")
print(f"  {'Cell types segregate' if assort > 0.1 else 'Cell types mix' if assort < -0.1 else 'Random mixing'}")

### Step 5: Topic Modeling

In [ ]:
print("\n" + "=" * 50)
print("SPATIAL LDA (TOPIC MODELING)")
print("=" * 50)

# Fit LDA
lda_model = fit_spatial_lda(tissue, n_topics=4, neighborhood_radius=50.0, random_state=42)

print(f"\nFitted {lda_model.n_topics} topics")

# Topic compositions
enrichment = topic_enrichment(lda_model, tissue)
print("\nTopic enrichment by cell type:")
print(enrichment.round(3))

### Step 6: Generate Report Figure

In [ ]:
# Create comprehensive figure
set_publication_style(journal='nature')

fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 4, hspace=0.35, wspace=0.35)

# Row 1: Spatial overview
ax1 = fig.add_subplot(gs[0, :2])
plot_spatial_scatter(tissue, ax=ax1)
ax1.set_title('A. Tissue Overview', loc='left', fontweight='bold')

ax2 = fig.add_subplot(gs[0, 2])
# Diversity map
scatter = ax2.scatter(tissue.coordinates[:, 0], tissue.coordinates[:, 1],
                       c=diversity, cmap='viridis', s=15)
plt.colorbar(scatter, ax=ax2, label='Shannon Diversity')
ax2.set_title('B. Neighborhood Diversity', loc='left', fontweight='bold')

ax3 = fig.add_subplot(gs[0, 3])
# Topic assignments
topics = dominant_topic_per_cell(lda_model, tissue)
scatter = ax3.scatter(tissue.coordinates[:, 0], tissue.coordinates[:, 1],
                       c=topics, cmap='Set1', s=15)
plt.colorbar(scatter, ax=ax3, label='Topic')
ax3.set_title('C. Spatial Topics', loc='left', fontweight='bold')

# Row 2: Statistics
ax4 = fig.add_subplot(gs[1, 0])
plot_ripleys_curve(tissue, ax=ax4, max_radius=150)
ax4.set_title('D. Clustering Analysis', loc='left', fontweight='bold')

ax5 = fig.add_subplot(gs[1, 1])
plot_colocalization_heatmap(clq, ax=ax5, annot=True)
ax5.set_title('E. Colocalization', loc='left', fontweight='bold')

ax6 = fig.add_subplot(gs[1, 2:])
# Cell type counts
counts = [np.sum(tissue.cell_types == ct) for ct in tissue.cell_types_unique]
ax6.bar(range(len(counts)), counts, color=plt.cm.tab10.colors[:len(counts)])
ax6.set_xticks(range(len(counts)))
ax6.set_xticklabels(tissue.cell_types_unique, rotation=45, ha='right')
ax6.set_ylabel('Cell Count')
ax6.set_title('F. Cell Type Distribution', loc='left', fontweight='bold')

# Row 3: Advanced analysis
ax7 = fig.add_subplot(gs[2, :2])
# Interface cells visualization
ax7.scatter(tissue.coordinates[:, 0], tissue.coordinates[:, 1],
            c='lightgray', s=10, alpha=0.3)
ax7.scatter(tissue.coordinates[tumor_int, 0], tissue.coordinates[tumor_int, 1],
            c='red', s=30, label=f'Tumor interface (n={len(tumor_int)})')
ax7.scatter(tissue.coordinates[cd8_int, 0], tissue.coordinates[cd8_int, 1],
            c='blue', s=30, label=f'CD8 interface (n={len(cd8_int)})')
ax7.legend()
ax7.set_title('G. Tumor-Immune Interface', loc='left', fontweight='bold')

ax8 = fig.add_subplot(gs[2, 2:])
# Topic composition heatmap
im = ax8.imshow(enrichment.values, cmap='YlOrRd', aspect='auto')
ax8.set_xticks(range(len(enrichment.columns)))
ax8.set_xticklabels([f'Topic {i}' for i in range(len(enrichment.columns))])
ax8.set_yticks(range(len(enrichment.index)))
ax8.set_yticklabels(enrichment.index)
plt.colorbar(im, ax=ax8, label='Mean Weight')
ax8.set_title('H. Topic-Cell Type Association', loc='left', fontweight='bold')

plt.tight_layout()
plt.show()

# Reset style
import matplotlib
matplotlib.rcdefaults()

## Workflow 2: Cohort Comparison Pipeline

This workflow compares multiple samples systematically.

In [ ]:
# Create cohort
def create_cohort(n_responders=6, n_nonresponders=6):
    """Create a cohort of samples."""
    samples = []
    sample_ids = []
    groups = []
    
    for i in range(n_responders):
        # Responders: high immune infiltration
        n_tumor = np.random.randint(150, 200)
        n_immune = np.random.randint(80, 120)  # More immune
        n_stromal = np.random.randint(40, 80)
        
        tumor = np.random.normal([500, 500], 80, (n_tumor, 2))
        immune = np.random.normal([500, 500], 120, (n_immune, 2))  # More infiltrating
        stromal = np.random.uniform(0, 1000, (n_stromal, 2))
        
        coords = np.vstack([tumor, immune, stromal])
        types = ['Tumor']*n_tumor + ['CD8_T_cell']*n_immune + ['Stromal']*n_stromal
        
        samples.append(SpatialTissueData(coordinates=coords, cell_types=np.array(types)))
        sample_ids.append(f'R{i+1}')
        groups.append('Responder')
    
    for i in range(n_nonresponders):
        # Non-responders: low/excluded immune
        n_tumor = np.random.randint(200, 280)
        n_immune = np.random.randint(20, 50)  # Less immune
        n_stromal = np.random.randint(60, 100)
        
        tumor = np.random.normal([500, 500], 100, (n_tumor, 2))
        immune = np.random.normal([500, 500], 250, (n_immune, 2))  # More peripheral
        stromal = np.random.uniform(0, 1000, (n_stromal, 2))
        
        coords = np.vstack([tumor, immune, stromal])
        types = ['Tumor']*n_tumor + ['CD8_T_cell']*n_immune + ['Stromal']*n_stromal
        
        samples.append(SpatialTissueData(coordinates=coords, cell_types=np.array(types)))
        sample_ids.append(f'NR{i+1}')
        groups.append('Non-Responder')
    
    return samples, sample_ids, groups

samples, sample_ids, groups = create_cohort()
print(f"Created cohort with {len(samples)} samples")

In [ ]:
# Define comprehensive panel
panel = StatisticsPanel(name='comprehensive')

# Population
panel.add('cell_counts')
panel.add('cell_proportions')

# Spatial
panel.add('ripleys_h_max', max_radius=100)
panel.add('mean_nearest_neighbor_distance')

# Neighborhood
panel.add('mean_neighborhood_entropy', radius=50.0)
panel.add('neighborhood_homogeneity', radius=50.0)

# Colocalization
panel.add('colocalization_score', type_a='CD8_T_cell', type_b='Tumor', radius=50.0)

# Compute summaries
multi_summary = MultiSampleSummary(samples, panel, sample_ids=sample_ids)
df = multi_summary.to_dataframe()
df['group'] = groups

print(f"Feature matrix: {df.shape}")
print(f"Features: {list(df.columns)}")

In [ ]:
# Statistical comparison
from scipy import stats

print("\n" + "=" * 50)
print("GROUP COMPARISON")
print("=" * 50)

numeric_cols = df.select_dtypes(include=[np.number]).columns

print("\nSignificant differences (p < 0.05):")
for col in numeric_cols:
    resp = df[df['group'] == 'Responder'][col].dropna()
    non_resp = df[df['group'] == 'Non-Responder'][col].dropna()
    
    if len(resp) > 2 and len(non_resp) > 2:
        _, p = stats.mannwhitneyu(resp, non_resp, alternative='two-sided')
        if p < 0.05:
            print(f"  {col}: p={p:.4f}")
            print(f"    Responder: {resp.mean():.3f} ± {resp.std():.3f}")
            print(f"    Non-Responder: {non_resp.mean():.3f} ± {non_resp.std():.3f}")

In [ ]:
# PCA visualization
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Prepare data
feature_cols = [c for c in df.columns if c not in ['group', 'sample_id']]
X = df[feature_cols].values
X = np.nan_to_num(X, nan=0.0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
colors = {'Responder': 'green', 'Non-Responder': 'red'}
for group in ['Responder', 'Non-Responder']:
    mask = np.array(groups) == group
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=colors[group], s=100, label=group, alpha=0.7)

for i, sid in enumerate(sample_ids):
    ax.annotate(sid, (X_pca[i, 0], X_pca[i, 1]), fontsize=8)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title('Cohort PCA: Spatial Features')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Workflow 3: Custom Analysis Pipeline

Template for creating your own analysis pipeline.

In [ ]:
def analyze_sample(tissue, sample_id='sample'):
    """
    Complete analysis pipeline for a single sample.
    
    Parameters
    ----------
    tissue : SpatialTissueData
        Input tissue data.
    sample_id : str
        Sample identifier.
    
    Returns
    -------
    dict
        Analysis results.
    """
    results = {'sample_id': sample_id}
    
    # 1. Basic QC
    results['n_cells'] = tissue.n_cells
    results['n_types'] = len(tissue.cell_types_unique)
    
    # 2. Cell counts
    for ct in tissue.cell_types_unique:
        results[f'n_{ct}'] = np.sum(tissue.cell_types == ct)
        results[f'prop_{ct}'] = results[f'n_{ct}'] / tissue.n_cells
    
    # 3. Spatial statistics
    radii = np.linspace(10, 100, 10)
    for ct in tissue.cell_types_unique:
        mask = tissue.cell_types == ct
        coords = tissue.coordinates[mask]
        if len(coords) >= 10:
            H = ripleys_h(coords, radii=radii)
            results[f'ripleys_h_max_{ct}'] = H.max()
    
    # 4. Neighborhood diversity
    neighborhoods = compute_neighborhoods(tissue, method='radius', radius=50.0)
    diversity = neighborhood_diversity(tissue, neighborhoods, metric='shannon')
    results['mean_diversity'] = diversity.mean()
    
    # 5. Network assortativity
    try:
        graph = CellGraph.from_spatial_data(tissue, method='proximity', radius=50.0)
        results['type_assortativity'] = type_assortativity(graph)
    except:
        results['type_assortativity'] = np.nan
    
    # 6. Topic diversity (if multiple types)
    if len(tissue.cell_types_unique) >= 3:
        try:
            lda = fit_spatial_lda(tissue, n_topics=3, neighborhood_radius=50.0, random_state=42)
            topics = dominant_topic_per_cell(lda, tissue)
            results['topic_entropy'] = -np.sum([
                (p := np.mean(topics == i)) * np.log(p + 1e-10)
                for i in range(3)
            ])
        except:
            results['topic_entropy'] = np.nan
    
    return results

# Apply to sample
analysis = analyze_sample(tissue, 'TME_001')
print("\nAnalysis Results:")
for key, value in analysis.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

## Best Practices Summary

### 1. Analysis Design
- Start with QC and visualization
- Define analysis goals upfront
- Choose appropriate neighborhood radius based on biology

### 2. Feature Selection
- Include population, spatial, and neighborhood metrics
- Consider biological interpretability
- Test multiple scales (radii)

### 3. Reproducibility
- Set random seeds
- Document parameter choices
- Save intermediate results

### 4. Reporting
- Create comprehensive figures
- Include statistical tests
- Export for downstream ML

## Congratulations!

You've completed all 12 spatialtissuepy tutorials. You now have the skills to:

1. Load and manage spatial tissue data
2. Compute spatial statistics (Ripley's K, colocalization, hotspots)
3. Analyze neighborhoods and build cell graphs
4. Discover tissue microenvironments with Spatial LDA and Mapper
5. Compare cohorts and extract ML features
6. Analyze ABM simulations
7. Create publication-quality figures

**Next steps:**
- Apply these methods to your own data
- Explore the API documentation for advanced options
- Contribute to the project on GitHub